In [44]:
import pandas as pd
import peptides

# Load the CSV files
df = pd.read_csv('../data/processed/dataset_80_20_normalized_tcr_with_cohort.csv')
df2 = pd.read_csv('../data/interim/prediction_pmtNet.csv')
df3 = pd.read_csv('../data/interim/prediction_pmtNet_nulls.csv')


In [45]:
df.head()

,Cancer_Type,gene,Peptide,wt_seq,pep_mut_start,mutant_best_alleles,mutant_rank,mutant_rank_netMHCpan,mutant_rank_PRIME,mut_Rank_Stab,...,immunogenicity,foreignness_score,hydrophobicity_avg_diff,hydrophobicity_total_diff,mut_Thalf,mw_tcr_contact,hydroph_tcr_contact,charge_tcr_contact,mutant_best_alleles_normalized,cohort
0,Melanoma,CHD5,KSKKEFRAY,KSEKEFRAY,3.0,HLA-C12:03,2.947,3.6904,2.947,0.8,...,0,0.483,-0.044,-0.4,1.6604,578.66894,-2.275,0.994865,C1203,HiTIDE
1,Melanoma,MYOCD,QSFSSTSAL,QSSSSTSAL,3.0,HLA-B07:02,1.779,1.8105,1.779,3.0,...,0,0.655,0.400,3.6,0.3099,380.35494,-0.775,-0.005094,B0702,NaN
2,Melanoma,GSE1,QAPDPAYIY,RAPDPAYIY,1.0,HLA-C07:02,0.039,0.0507,0.039,1.8,...,0,0.366,0.111,1.0,0.1162,464.47534,-1.150,-1.007049,C0702,NCI
3,Melanoma,SUPT16H,GSDAEEGDSA,GSDAEEGDSE,10.0,HLA-A01:01,4.563,8.1864,4.563,42.0,...,0,0.669,0.530,5.3,0.1289,519.46554,-1.820,-3.003501,A0101,HiTIDE
4,Melanoma,FGF7,WYLRINKRGKV,WYLRIDKRGKV,6.0,HLA-A24:02,22.955,0.5629,22.955,3.5,...,0,0.379,0.000,0.0,0.2800,742.87944,-2.050,2.994149,A2402,NaN


In [46]:
# En df2 me quedo con las combinaciones de Antigen HLA que tengan el mayor Rank.

df2 = df2.loc[df2.groupby(['Antigen', 'HLA'])['Rank'].idxmin()]
df2.reset_index(drop=True, inplace=True)

In [47]:
df2 

,CDR3,Antigen,HLA,Rank
0,CASSTMGNAPLDTQYF,AAAAAAQQIQV,B*07:02,0.0000
1,CASSLHLLRGLNTGELFF,AAAAAGVSFTK,A*11:01,0.0144
2,CASSQGGDYGYTF,AAAAGVSFTK,A*11:01,0.0178
3,CASGDSGDEQFF,AAAAQQIQV,A*02:01,0.0042
4,CAGGSYNNNDMRF,AAAEKLPSL,B*07:02,0.0020
...,...,...,...,...
8165,CAVGGQKLGDTQYF,YYPGKKEPF,A*24:02,0.0410
8166,CSARGLAEANTGELFF,YYQTETYDL,A*24:07,0.0078
8167,CAVGGQKLGDTQYF,YYSPQAQTL,A*24:02,0.0141
8168,CASSEDSGGYNEQFF,YYTNLEQKL,C*06:02,0.0021


In [48]:
#Todos los valores negativos en Rank los cambio a 0
df2['Rank'] = df2['Rank'].apply(lambda x: max(x, 0))

In [49]:
#Lo mismo pero con df3
df3 = df3.loc[df3.groupby(['Antigen', 'HLA'])['Rank'].idxmin()]
df3.reset_index(drop=True, inplace=True)


In [50]:
df3

,CDR3,Antigen,HLA,Rank
0,CASSLHLLRGLNTGELFF,AAAAAGVSFTK,A*11:01,0.0144
1,CASGDRTGGVGYTF,AAAAVTPGL,B*35:01,0.0055
2,CASSQGGDYGYTF,AAAGVSFTK,A*11:01,0.0176
3,CASSQSPGGTQYF,AAASATLAL,C*08:01,0.0205
4,CASSLHLLRGLNTGELFF,AAHIIMIFEK,A*11:01,0.0106
...,...,...,...,...
2109,CSARGLAEANTGELFF,YYQGRRFTI,A*24:02,0.0285
2110,CASSSHDRQGQNSPLHF,YYSKNLNSF,A*24:02,0.0560
2111,CASSSHDRQGQNSPLHF,YYSKNLNSF,A*24:03,0.0560
2112,CASSVLAGGYNEQFF,YYSKNLNSFF,A*24:02,0.0040


In [51]:
#Todos los valores negativos en Rank los cambio a 0
df3['Rank'] = df3['Rank'].apply(lambda x: max(x, 0))

# Ahora uno df2 y df3 para tener un solo dataframe con todas las predicciones
df2 = pd.concat([df2, df3], ignore_index=True)

In [52]:
df2

,CDR3,Antigen,HLA,Rank
0,CASSTMGNAPLDTQYF,AAAAAAQQIQV,B*07:02,0.0000
1,CASSLHLLRGLNTGELFF,AAAAAGVSFTK,A*11:01,0.0144
2,CASSQGGDYGYTF,AAAAGVSFTK,A*11:01,0.0178
3,CASGDSGDEQFF,AAAAQQIQV,A*02:01,0.0042
4,CAGGSYNNNDMRF,AAAEKLPSL,B*07:02,0.0020
...,...,...,...,...
10279,CSARGLAEANTGELFF,YYQGRRFTI,A*24:02,0.0285
10280,CASSSHDRQGQNSPLHF,YYSKNLNSF,A*24:02,0.0560
10281,CASSSHDRQGQNSPLHF,YYSKNLNSF,A*24:03,0.0560
10282,CASSVLAGGYNEQFF,YYSKNLNSFF,A*24:02,0.0040


In [53]:
# 1. Limpiamos el formato en df 
# Eliminamos "HLA-" y también los dos puntos ":"
# Ejemplos: 
# "HLA-B07:02" -> "B0702"
# "B1501"      -> "B1501" (no cambia, pero ya coincide)
df['hla_clean'] = df['mutant_best_alleles'].str.replace('HLA-', '', regex=False).str.replace(':', '', regex=False)

# 2. Limpiamos el formato en df2
# Eliminamos el asterisco "*" y también los dos puntos ":"
# Ejemplo: "B*07:02" -> "B0702"
df2['hla_clean'] = df2['HLA'].str.replace('*', '', regex=False).str.replace(':', '', regex=False)

# 3. Hacemos el merge (esto se mantiene igual, pero ahora los datos coinciden)
df_final = pd.merge(
    df, 
    df2[['Antigen', 'hla_clean', 'Rank']], 
    left_on=['Peptide', 'hla_clean'],      
    right_on=['Antigen', 'hla_clean'],     
    how='left'
)

# 4. Verificamos
print(f"Nulos tras corrección: {df_final['Rank'].isnull().sum()}")
df_final.head()

Nulos tras corrección: 0


,Cancer_Type,gene,Peptide,wt_seq,pep_mut_start,mutant_best_alleles,mutant_rank,mutant_rank_netMHCpan,mutant_rank_PRIME,mut_Rank_Stab,...,hydrophobicity_total_diff,mut_Thalf,mw_tcr_contact,hydroph_tcr_contact,charge_tcr_contact,mutant_best_alleles_normalized,cohort,hla_clean,Antigen,Rank
0,Melanoma,CHD5,KSKKEFRAY,KSEKEFRAY,3.0,HLA-C12:03,2.947,3.6904,2.947,0.8,...,-0.4,1.6604,578.66894,-2.275,0.994865,C1203,HiTIDE,C1203,KSKKEFRAY,0.0087
1,Melanoma,MYOCD,QSFSSTSAL,QSSSSTSAL,3.0,HLA-B07:02,1.779,1.8105,1.779,3.0,...,3.6,0.3099,380.35494,-0.775,-0.005094,B0702,NaN,B0702,QSFSSTSAL,0.0036
2,Melanoma,GSE1,QAPDPAYIY,RAPDPAYIY,1.0,HLA-C07:02,0.039,0.0507,0.039,1.8,...,1.0,0.1162,464.47534,-1.150,-1.007049,C0702,NCI,C0702,QAPDPAYIY,0.0022
3,Melanoma,SUPT16H,GSDAEEGDSA,GSDAEEGDSE,10.0,HLA-A01:01,4.563,8.1864,4.563,42.0,...,5.3,0.1289,519.46554,-1.820,-3.003501,A0101,HiTIDE,A0101,GSDAEEGDSA,0.0155
4,Melanoma,FGF7,WYLRINKRGKV,WYLRIDKRGKV,6.0,HLA-A24:02,22.955,0.5629,22.955,3.5,...,0.0,0.2800,742.87944,-2.050,2.994149,A2402,NaN,A2402,WYLRINKRGKV,0.0047


In [54]:
# Observo todos los nulos en la columna Rank
nulos = df_final[df_final['Rank'].isnull()]
nulos

,Cancer_Type,gene,Peptide,wt_seq,pep_mut_start,mutant_best_alleles,mutant_rank,mutant_rank_netMHCpan,mutant_rank_PRIME,mut_Rank_Stab,...,hydrophobicity_total_diff,mut_Thalf,mw_tcr_contact,hydroph_tcr_contact,charge_tcr_contact,mutant_best_alleles_normalized,cohort,hla_clean,Antigen,Rank


In [55]:
#Compruebo cuantos valores nulos hay en la columna Rank
df_final['Rank'].isnull().sum()


0

In [56]:
# Guardo el dataframe final en un nuevo CSV
df_final.to_csv('dataset_80_20_normalized_tcr_with_cohort_and_rank.csv', index=False)